# v112_france_rules — v110 + French normalisation fixes (rules v4), France re-run

| Field | Value |
|---|---|
| **Version** | `v112_france_rules` |
| **Plan group** | B2 / B3 (legal-form and abbreviation normalisation) |
| **Parent version** | v110 (`v110_m3_features`) |
| **Author** | M1 rajaguru2004 |
| **Date** | 2026-09-26 |
| **Status** | shortlisted |

France is 15 % of the test's Source-1 entities (259k) and absent from train, so no local fold
can score it. A read-only audit of the French test records (S1 259k, S2 703k, S3 732k)
against the rules-v3 normalisation found three systematic differences between Source 1 and
the pool that touch no US or Indian record:

* the number marker **"N°" / "Nº"**: 6.6 % of French pool addresses, never in S1; folding
  turned it into a stray `n` token (+9,259 look-alike pairs become token-identical);
* **"bis"** written `B` by the pool (`12 bis` vs `12B`; +997 identical);
* **"compagnie"** vs the legal form `cie` (+374 name matches).

Rules v4 add exactly these three. They change no US or Indian text (checked record by record
on the test split), so this version re-runs **only France** and reuses the parent's cached
India and US test outputs: same stage 1, stage 2 and rule as the parent.

## 1. Hypothesis

* **Change vs parent:** normalisation rules v4 (French number marker, `bis`, `compagnie`);
  France's test candidates, features and scores are recomputed, nothing else.
* **Why:** a stray `n` token and `bis`/`B` lower address similarity for ~7 % of French true
  pairs; the model sees them as weaker matches than they are.
* **Check:** India and US outputs must be identical to the parent's (they are not
  recomputed and their normalised text does not change); France's candidate set, matched
  share and matches per S1 are compared with the parent's. The leaderboard is the only
  score for France: the upload measures it.
* **Discard if:** the public score does not beat the parent's.

## 2. Setup

The parent's configuration and fitted two-stage model come back from its artifacts. The
stage-1 test cache of this version starts as a copy of the parent's India and US files, so
`run_test_two_stage` recomputes only France.

In [ ]:
import json
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

from entity_resolution import config as C
from entity_resolution.normalize import RULES_VERSION
from entity_resolution.pipeline import PipelineConfig, peak_rss_gb
from entity_resolution.data import isin, load_source
from entity_resolution.tracking import log_result
from entity_resolution.twostage import TwoStage, run_test_two_stage

pd.set_option("display.width", 200)

EXP_DIR = C.EXPERIMENTS / "v112_france_rules"
ARTIFACTS = EXP_DIR / "artifacts"
PARENT_DIR = C.EXPERIMENTS / "v110_m3_features"
parent = json.loads((PARENT_DIR / "metrics.json").read_text())
cfg = PipelineConfig.from_record(json.loads((PARENT_DIR / "artifacts" / "stage1" /
                                            "config.json").read_text()))
ts = TwoStage.load(PARENT_DIR / "artifacts", cfg)
assert RULES_VERSION == 4, RULES_VERSION
# the parent's stage-1 test outputs (v111 reads v110's): one parquet per country
PARENT_CACHE = Path(parent["metrics"]["stage1_cache"]) / "test"
CACHE = cfg.cache_dir / "stage1" / (PARENT_CACHE.parent.name + "_rules4") / "test"
CACHE.mkdir(parents=True, exist_ok=True)
for f in PARENT_CACHE.iterdir():
    if not f.name.startswith("test_France") and not (CACHE / f.name).exists():
        shutil.copy2(f, CACHE / f.name)
timings: dict[str, float] = {}
t_start = time.time()
print("parent", PARENT_DIR.name, "| rules", RULES_VERSION, "| cache", CACHE)
sorted(p.name for p in CACHE.iterdir())

## 3. Data

The test split only (the mock fold has no French entity): Source 1 normalised under rules v4,
France's pool normalised and blocked under rules v4 inside `run_test_two_stage`.

## 4. Method

`run_test_two_stage` with the parent's model: India and US come from the copied cache,
France is computed (normalise, block, features, stage 1 + filter, stage 2), then the parent's
rule decides every country. Both validators run on the files.

In [ ]:
t0 = time.time()
match_path, cand_path, s1n_test, test_matches, test_summary = run_test_two_stage(
    cfg, ts, timings=timings, cache_dir=CACHE)
timings["run_test_seconds"] = round(time.time() - t0, 2)
dest = C.ROOT / "submissions" / "v112"
dest.mkdir(parents=True, exist_ok=True)
for p in (match_path, cand_path):
    shutil.copy2(p, dest / p.name)
out = subprocess.run([sys.executable, "-m", "entity_resolution.submission", "--output-dir",
                      str(C.OUTPUT), "--check-ids"], capture_output=True, text=True)
print(out.stdout[-2000:], out.stderr[-2000:])
out = subprocess.run([sys.executable, str(C.OFFICIAL_VALIDATOR), "--matching", str(match_path),
                      "--candidate", str(cand_path), "--test-dir", str(C.DATASET / "test")],
                     capture_output=True, text=True)
print(out.stdout[-3000:], out.stderr[-2000:])
print(f"run_test {timings['run_test_seconds']:.0f} s")

## 5. Evaluation

Per country, this version against the parent's submitted files: India and US must be
identical; France's candidates and matches show what rules v4 changed.

In [ ]:
country_of = s1n_test.set_index(C.ENTITY_ID)[C.COUNTRY]
parent_dest = C.ROOT / "submissions" / "v110"


def rows_of(path: Path) -> pd.DataFrame:
    """A submission file as written: one row per S1 id, the pool ids comma-joined."""
    return pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False)


def pairs_of(frame: pd.DataFrame) -> pd.DataFrame:
    """(S1 id, pool id) pairs of a submission frame, one row per listed pool id."""
    col = frame.columns[1]
    f = frame.assign(**{col: frame[col].str.split(",")}).explode(col)
    return f[f[col] != ""].rename(columns={col: "pool_id"}).reset_index(drop=True)


def per_country(d: Path) -> pd.DataFrame:
    """S1 count, candidates per S1, matched share and matches per S1, by country."""
    m = pairs_of(rows_of(d / C.MATCHING_FILE))
    c = pairs_of(rows_of(d / C.CANDIDATE_FILE))
    n_s1 = country_of.value_counts()
    return pd.DataFrame({
        "s1": n_s1,
        "cands_per_s1": c[C.S1_ID].map(country_of).value_counts() / n_s1,
        "matched_share": m.drop_duplicates(C.S1_ID)[C.S1_ID].map(country_of).value_counts()
                         / n_s1,
        "matches_per_s1": m[C.S1_ID].map(country_of).value_counts() / n_s1})


cmp = pd.concat({"v112": per_country(dest), "v110": per_country(parent_dest)},
                axis=1).round(4)
identical = {}
for name in (C.MATCHING_FILE, C.CANDIDATE_FILE):
    a, b = rows_of(dest / name), rows_of(parent_dest / name)
    keep = (a[C.S1_ID].map(country_of) != "France").to_numpy()
    identical[name] = bool(a[keep].reset_index(drop=True).equals(
        b[(b[C.S1_ID].map(country_of) != "France").to_numpy()].reset_index(drop=True)))
fr = {}
for tag, d in (("v112", dest), ("parent", parent_dest)):
    m = pairs_of(rows_of(d / C.MATCHING_FILE))
    fr[tag] = m[(m[C.S1_ID].map(country_of) == "France").to_numpy()]
diff = fr["v112"].merge(fr["parent"], how="outer", indicator=True)["_merge"].value_counts()
print("India/US rows identical to the parent:", identical)
print("France pairs: only v112", diff.get("left_only", 0), "| only parent",
      diff.get("right_only", 0), "| both", diff.get("both", 0))
cmp

## 6. Error analysis

France has no labels; a sample of the French pairs that only this version predicts, raw
records side by side, shows what the rules changed.

In [ ]:
new_only = (fr["v112"].merge(fr["parent"], how="left", indicator=True)
            .query("_merge == 'left_only'").drop(columns="_merge"))
sample = new_only.sample(min(12, len(new_only)), random_state=0) if len(new_only) else new_only
cols = [C.ENTITY_ID, C.NAME, C.ADDRESS]
s1_raw = load_source("test", 1, cfg.dataset_dir, columns=cols)
s1_raw = s1_raw[isin(s1_raw[C.ENTITY_ID], pd.Index(sample[C.S1_ID]))]
pool_raw = pd.concat([load_source("test", k, cfg.dataset_dir, columns=cols) for k in (2, 3)],
                     ignore_index=True)
pool_raw = pool_raw[isin(pool_raw[C.ENTITY_ID], pd.Index(sample["pool_id"]))]
view = (sample.merge(s1_raw.rename(columns={C.ENTITY_ID: C.S1_ID}), on=C.S1_ID)
        .merge(pool_raw.rename(columns={C.ENTITY_ID: "pool_id"}), on="pool_id",
               suffixes=("_s1", "_pool")))
del s1_raw, pool_raw
print(len(new_only), "French pairs only in v112")
view

## 7. Log the result

The mock fold has no French entity, so its score is the parent's by construction; the
version is judged on the leaderboard.

In [ ]:
record = {
    "hypothesis": "French number marker, bis and compagnie normalisation (rules v4) lift France",
    "rules_version": RULES_VERSION, "parent": PARENT_DIR.name, "cache": str(CACHE),
    "per_country": cmp.to_dict(), "non_france_identical": identical,
    "france_pairs_only_new": int(diff.get("left_only", 0)),
    "france_pairs_only_parent": int(diff.get("right_only", 0)),
    "france_pairs_both": int(diff.get("both", 0)),
    "est_public_parent": parent["metrics"].get("est_public"), **timings,
    "peak_rss_gb": peak_rss_gb(),
}
assert all(identical.values()), "India/US outputs changed: rules v4 is not France-only"
row = log_result(
    EXP_DIR, change="v110 + French normalisation rules v4 (N° marker, bis, compagnie); France re-run",
    group="B2", mock_f05=float(parent["mock_f05"]) if parent.get("mock_f05") else None,
    notes=(f"mock = parent's (no France in the mock); France pairs +{record['france_pairs_only_new']} "
           f"/ -{record['france_pairs_only_parent']}; India/US identical"),
    metrics=record, owner="M1", parent="v110", decision="")
row

## 8. Conclusion

Written after the upload from the public score.

## 9. Test inference

Section 4 is the test inference: files in `submissions/v112/` and `output/`, both validators
above.